In [77]:
import pandas as pd
from pyproj import Transformer
import warnings
import xarray as xr
from itertools import combinations
import numpy as np

In [78]:
warnings.filterwarnings("ignore")

#### Read the raw file obtained from the EPA AQS data

##### Do coordinate transformation

In [79]:
input_file = r"\data\aqs_data_10_years\hourly_88101_2023.csv"
df_input = pd.read_csv(input_file)

In [80]:
filename = input_file.split("\\")[-1].replace(".csv", "")
year = filename[-4:]
print(year)

2023


In [81]:
df_input.head()

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Date Local,...,Units of Measure,MDL,Uncertainty,Qualifier,Method Type,Method Code,Method Name,State Name,County Name,Date of Last Change
0,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2023-01-09,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-04-30
1,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2023-01-09,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-04-30
2,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2023-01-09,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-04-30
3,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2023-01-09,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-04-30
4,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2023-01-09,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-04-30


In [82]:
df_input = df_input[["POC", "Latitude", "Longitude", "Datum", "Date GMT", "Time GMT", "Sample Measurement"]]
df_input.rename(columns={'Sample Measurement': 'pm25'}, inplace=True)
df_input.head()

,POC,Latitude,Longitude,Datum,Date GMT,Time GMT,pm25
0,3,30.497478,-87.880258,NAD83,2023-01-09,16:00,6.7
1,3,30.497478,-87.880258,NAD83,2023-01-09,17:00,5.6
2,3,30.497478,-87.880258,NAD83,2023-01-09,18:00,10.7
3,3,30.497478,-87.880258,NAD83,2023-01-09,19:00,6.0
4,3,30.497478,-87.880258,NAD83,2023-01-09,20:00,4.1


In [83]:
# Filter unique NAD83 lat/lon pairs
nad83_coords = df_input[df_input['Datum'] == 'NAD83'][['Latitude', 'Longitude']].drop_duplicates()
# Set up the transformer
transformer = Transformer.from_crs("EPSG:4269", "EPSG:4326", always_xy=True)
#Define a function to convert a lat/lon pair
def convert(lat, lon):
    lon_wgs, lat_wgs = transformer.transform(lon, lat)
    return pd.Series([lat_wgs, lon_wgs])
#Apply transformation to the unique NAD83 pairs
nad83_coords[['Latitude_wgs', 'Longitude_wgs']] = nad83_coords.apply(
    lambda row: convert(row['Latitude'], row['Longitude']), axis=1
)
#Merge converted values back to the original DataFrame
df_input = df_input.merge(nad83_coords, on=['Latitude', 'Longitude'], how='left')
#Replace NAD83 values with WGS84 values
df_input['Latitude'] = df_input.apply(
    lambda row: row['Latitude_wgs'] if row['Datum'] == 'NAD83' else row['Latitude'],
    axis=1
)
df_input['Longitude'] = df_input.apply(
    lambda row: row['Longitude_wgs'] if row['Datum'] == 'NAD83' else row['Longitude'],
    axis=1
)
df_input['Datum'] = 'WGS'
df_input.drop(columns=['Latitude_wgs', 'Longitude_wgs'], inplace=True)

#### Export the input (after converting to WGS)

In [84]:
df_input.to_csv(rf'\data\aqs_data_10_years\{year}.csv')

##### Extract unique combinations of each lat/lng

In [21]:
unique_coords = df_input[['Latitude', 'Longitude', 'Datum']].drop_duplicates()
datum_counts = unique_coords['Datum'].value_counts()
points_counts = df_input['Datum'].value_counts()
total_unique_count = unique_coords.shape
# Print all results
print("Unique coordinate combinations:")
print(total_unique_count)
print("\nUnique counts by datum:")
print(datum_counts)
print("\nDatum points frequency counts:")
print(points_counts)

Unique coordinate combinations:
(863, 3)

Unique counts by datum:
Datum
WGS    863
Name: count, dtype: int64

Datum points frequency counts:
Datum
WGS    9340031
Name: count, dtype: int64


#### Keep only files that fall in the US 48 region

In [22]:
ds = xr.open_dataset(r'/data/nldas_subset_data/NLDAS_FORA0125_H.A20230102.0900.002.grb.SUB.nc4')
lat_min, lat_max = ds['lat'].min().item(), ds['lat'].max().item()
lon_min, lon_max = ds['lon'].min().item(), ds['lon'].max().item()

df_contiguous_us = unique_coords[
    (unique_coords['Latitude'].between(lat_min, lat_max)) &
    (unique_coords['Longitude'].between(lon_min, lon_max))
].copy()

In [24]:
df_contiguous_us.shape

(844, 3)

#### Near Duplicates Check

In [25]:
tolerance = 1e-3
# Get all unique coordinate pairs
coords = list(zip(df_contiguous_us['Latitude'], df_contiguous_us['Longitude']))
# Compare each pair of coordinates
near_duplicates = []
for (i, coord1), (j, coord2) in combinations(enumerate(coords), 2):
    if np.allclose(coord1, coord2, atol=tolerance):
        near_duplicates.append((i, j, coord1, coord2))
if near_duplicates:
    print("Near-duplicate coordinate pairs found (within tolerance):")
    for i, j, c1, c2 in near_duplicates:
        print(f"Row {i} and Row {j} --> {c1} ≈ {c2}")
else:
    print("No near-duplicate coordinates found within the given tolerance.")

No near-duplicate coordinates found within the given tolerance.


In [26]:
df_contiguous_us.to_csv(rf"data\coordinates\yearly_coordinates\{year}.csv", index=False)